In [42]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

import optuna

import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

df = pd.read_csv("Netflix Dataset.csv")

# Kolla datatyper
print(df.dtypes)
print(df.head())

ModuleNotFoundError: No module named 'optuna'

In [34]:
# Filmer: Duration innehåller "min"
movies = df[df['Duration'].str.contains("min", na=False)].copy()
movies['Minutes'] = movies['Duration'].str.extract(r'(\d+)').astype(float)
movies['Duration'] = movies['Duration'].str.replace(r'\d+', '', regex=True).str.strip()

# Serier: Duration innehåller "Season"
shows = df[df['Duration'].str.contains("Season", na=False)].copy()
shows['Seasons'] = shows['Duration'].str.extract(r'(\d+)').astype(float)
shows['Duration'] = shows['Duration'].str.replace(r'\d+', '', regex=True).str.strip()
# Fyll NaN med "Unknown" först
movies['Rating'] = movies['Rating'].fillna("Unknown")

# Skapa LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(movies['Rating'])

# Kolla mapping
print(dict(zip(le.classes_, range(len(le.classes_)))))

# Kontrollera resultat
print(f"Antal filmer: {movies.shape[0]}")
print(movies[['Title', 'Duration', 'Minutes']].head())

print(f"Antal serier: {shows.shape[0]}")
print(shows[['Title', 'Duration', 'Seasons']].head())

print("NaN i numeriska features:", movies[num_cols].isna().sum())
print("NaN i kategoriska features:", movies[cat_cols].isna().sum())
print("NaN i target:", y.isna().sum())


{'G': 0, 'NC-17': 1, 'NR': 2, 'PG': 3, 'PG-13': 4, 'R': 5, 'TV-14': 6, 'TV-G': 7, 'TV-MA': 8, 'TV-PG': 9, 'TV-Y': 10, 'TV-Y7': 11, 'TV-Y7-FV': 12, 'UR': 13, 'Unknown': 14}
Antal filmer: 5379
   Title Duration  Minutes
1  07:19      min     93.0
2  23:59      min     78.0
3      9      min     80.0
4     21      min    123.0
6    122      min     95.0
Antal serier: 2410
     Title Duration  Seasons
0       3%  Seasons      4.0
5       46   Season      1.0
11    1983   Season      1.0
12    1994   Season      1.0
16  Feb-09   Season      1.0
NaN i numeriska features: Minutes    0
dtype: int64
NaN i kategoriska features: Category      0
Rating        0
Country     230
Type          0
dtype: int64
NaN i target: 0


In [35]:

movies['Rating'] = movies['Rating'].fillna("Unknown")
movies['Country'] = movies['Country'].fillna("Unknown")
print(movies[['Rating', 'Country']].isna().sum())
# Text feature: Description
movies['Description'] = movies['Description'].fillna("")
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_desc = tfidf.fit_transform(movies['Description'])

# Kategoriska features
cat_cols = ['Category', 'Rating', 'Country', 'Type']
encoder = OneHotEncoder(handle_unknown='ignore')
cat_feats = encoder.fit_transform(movies[cat_cols].fillna('Unknown'))

# Numeriska features
num_cols = ['Minutes']
scaler = StandardScaler()
num_feats = scaler.fit_transform(movies[num_cols].fillna(0))

# Kombinera alla features
X = hstack([tfidf_desc, cat_feats, num_feats])

# Target
y = y_encoded  # Om vi vill göra klassificering



Rating     0
Country    0
dtype: int64


In [38]:
# 80% train_full, 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
# 25% val från train_full → 60% train, 20% val, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)


In [40]:
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# Definiera modeller
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    "MLP": MLPClassifier(hidden_layer_sizes=(128,64), max_iter=50, random_state=42)
}

# Träna och utvärdera på valideringsset
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds_val = model.predict(X_val)
    acc = accuracy_score(y_val, preds_val)
    results[name] = acc
    print(f"\n{name} – Valideringsresultat:")
    print(classification_report(y_val, preds_val, zero_division=0))

Train: 3227, Val: 1076, Test: 1076

RandomForest – Valideringsresultat:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           2       1.00      1.00      1.00        16
           3       1.00      1.00      1.00        50
           4       1.00      1.00      1.00        77
           5       1.00      1.00      1.00       133
           6       1.00      1.00      1.00       255
           7       1.00      1.00      1.00        22
           8       1.00      1.00      1.00       369
           9       1.00      1.00      1.00       101
          10       1.00      1.00      1.00        23
          11       1.00      1.00      1.00        19
          12       0.00      0.00      0.00         1
          13       0.00      0.00      0.00         1
          14       1.00      1.00      1.00         1

    accuracy                           1.00      1076
   macro avg       0.86      0.86      0.86      1076
weighted

c:\Users\SebbePwnYou\anaconda3\envs\Python\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:37:59] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost – Valideringsresultat:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           2       1.00      1.00      1.00        16
           3       1.00      1.00      1.00        50
           4       1.00      1.00      1.00        77
           5       1.00      1.00      1.00       133
           6       1.00      1.00      1.00       255
           7       1.00      1.00      1.00        22
           8       1.00      1.00      1.00       369
           9       1.00      1.00      1.00       101
          10       1.00      1.00      1.00        23
          11       1.00      1.00      1.00        19
          12       1.00      1.00      1.00         1
          13       1.00      1.00      1.00         1
          14       1.00      1.00      1.00         1

    accuracy                           1.00      1076
   macro avg       1.00      1.00      1.00      1076
weighted avg       1.00      1.00      1.00     

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    clf = XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
    scores = cross_val_score(clf, X_train, y_train, cv=3, scoring="accuracy")
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Bästa hyperparametrar:", study.best_params)
